In [2]:
from prophet import Prophet
import pandas as pd

/Users/tudormorariu/Documents/ai_work/.venv3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train = pd.read_csv("train_final.csv")
test = pd.read_csv('test_final.csv')

In [4]:
df = train.copy()
df["date"] = pd.to_datetime(df["date"])

# aggregate (VERY IMPORTANT)
ts = (
    df.groupby(["date", "store_id", "product_id"])["demand"]
    .sum()
    .reset_index()
)

In [5]:
ts

,date,store_id,product_id,demand
0,2023-05-15,CM-BAC-01,P0001,3
1,2023-05-15,CM-BAC-01,P0002,6
2,2023-05-15,CM-BAC-01,P0003,0
3,2023-05-15,CM-BAC-01,P0004,1
4,2023-05-15,CM-BAC-01,P0005,0
...,...,...,...,...
421075,2025-05-10,CM-TIM-01,P0054,1
421076,2025-05-10,CM-TIM-01,P0055,0
421077,2025-05-10,CM-TIM-01,P0056,0
421078,2025-05-10,CM-TIM-01,P0057,2


In [6]:
def train_prophet(group):

    data = group.rename(columns={"date": "ds", "demand": "y"})[["ds", "y"]]

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False
    )

    model.fit(data)

    return model

In [7]:
models = {}

for key, group in ts.groupby(["store_id", "product_id"]):

    models[key] = train_prophet(group)

13:28:39 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1] done processing
13:28:40 - cmdstanpy - INFO - Chain [1] start processing
13:28:40 - cmdstanpy - INFO - Chain [1]

In [8]:

def predict_future(model, periods):

    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)

    return forecast[["ds", "yhat"]] 

In [10]:
import tqdm
preds = []

for _, row in tqdm.tqdm(test.iterrows()):

    key = (row["store_id"], row["product_id"])

    model = models.get(key)

    if model is None:
        preds.append(0)  # fallback
        continue

    forecast = predict_future(model, periods=30)  # adjust horizon

    pred = forecast.iloc[-1]["yhat"]
    preds.append(pred)

0it [00:00, ?it/s]

374it [00:24, 15.49it/s]


KeyboardInterrupt: 